### 基本定义

RAG（Retrieval-Augmented Generation，检索增强生成）是一种将信息检索与文本生成相结合的AI技术。
例如：我们向LLM提问一个问题，RAG从各种数据源检索相关的信息，并将检索到的信息和问题注入到LLM提示中，LLM最后给出答案。

### 为什么有这个技术

当我们将大模型应用于实际业务场景时会发现，通用的基础大模型基本无法满足我们的实际业务需求，主要有以下几方面原因：

#### 知识的局限性

模型自身的知识完全源于它的训练数据，而现在的主流大模型的训练集基本都是源于网络公开的数据，对于一些实时性的、非公开的或离线的数据是无法获取到的，这部分知识也就无从具备。

#### 幻觉问题

所有AI模型的核心基本数据概率，输出本质是一系列数值运算，大模型也不例外。因为，在知识欠缺或不擅长的领域，大模型可能一本正经的输出错误信息。这种“幻觉”问题难以辨别，因为它要求使用者具备相关领域知识。

#### 数据安全性

对企业而言，数据安全至关重要，没有企业愿意冒数据泄露风险将私域数据上传至第三方平台进行训练。因此，完全依赖通用大模型的应用方案往往需要再数据安全与效果之间权衡。

### RAG的工作原理

RAG的核心思想是在LLM回答用户问题之前，先进行一次信息检索。这个过程通常分为以下几个主要步骤：
- 用户查询
- 知识库
- 检索
- 增强提示词
- 生成

想象你是一名学生，需要回答一道复杂的历史问题。
- 没有RAG：你只能凭记忆回答，如果问题超出你得记忆范围，你可能会胡编乱造。
- 有了RAG：你会先去图书馆（知识库）查找相关的历史书籍（检索），然后阅读书中的相关章节（增强），最后结合这些信息和自己的理解来回答问题（生成）。

### RAG系统搭建流程

- 收集数据
- 数据分块
- 选择文本嵌入模型
- 初始化向量数据库
- 存储向量
- 整合大预言模型（LLM）


In [1]:
# 收集数据
from langchain_community.document_loaders import TextLoader # 加载文档

# 加载文件
loader = TextLoader("./政策文件.txt", encoding='utf-8')
documents = loader.load()

In [2]:
# 数据分块
from langchain.text_splitter import RecursiveCharacterTextSplitter # 对文本进行拆分

# 中文分块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, # 适合中文段落（段落大小）
    chunk_overlap=0, # 避免上下文缺失(保持段落完整性)
    separators=["\n\n", "\n", "。", "！", "？"],
)
splits = text_splitter.split_documents(documents)

In [3]:
# 选择文本嵌入模型
from langchain_huggingface import HuggingFaceEmbeddings

# 加载本地模型
embeddings = HuggingFaceEmbeddings(
    model_name='tmp/bge-small-zh-v1.5',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': False} # 提升相似度计算精度
)

In [5]:
# 初始化向量数据库
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=splits,               # 输入的文本
    embedding=embeddings,           # 选择模型
    persist_directory="./tmp/chroma_db2" # 数据库存储的目录
)

# 语义搜索
docs = db.similarity_search("年假的有效期", k=1) # k是匹配的内容几段
print('匹配结果：')
for doc in docs:
    print(doc.page_content)

匹配结果：
公司年假政策：
1. 员工工作满1年可享受5天带薪年假
2. 年假有效期至次年3月31日


In [ ]:
# 存储数据

# 已在上面传入到Chroma了 // documents=splits

In [15]:
# 加载大语言模型

import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import warnings

warnings.filterwarnings("ignore", message=".*HMAC key is 16 bytes long.*")

key = os.getenv('ZHIPU_API_KEY')
os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)

In [17]:
# 结合LLM模型使用
from langchain.chains.retrieval_qa.base import RetrievalQA
# from langchain.chains import RetrievalQA 也可以这么简写

# 创建一个RAG链
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=db.as_retriever(search_kwargs={'k': 1}), # 检索几段内容
    return_source_documents=True # 返回源文档（调式用的）
)

# 测试
query = "年假有效期"
result = qa_chain({'query': query})
print('答案：', result['result'])
print('来源：', result['source_documents'][0].page_content)

答案： 根据公司年假政策，年假的有效期至次年3月31日。这意味着员工未能在当年使用的年假，最晚可以在次年的3月31日前使用完毕。
来源： 公司年假政策：
1. 员工工作满1年可享受5天带薪年假
2. 年假有效期至次年3月31日


### 练习

In [8]:
#! pip install pymupdf==1.27.2.2
import fitz

def extract_text_pdf(pdf, start=1, end=None):
    """
    加载PDF文件的内容
    :param pdf:  PDF
    :param start: 开始页
    :param end: 结束页
    :return: 返回文件内容和页码数据
    """
    text = ''
    page_numbers = []
    for pn, page in enumerate(pdf):
        if pn < start:
            continue
        if end is not None and pn == end:
            break
        # 读取数据
        extract_text = page.get_text('text')
        # 判断是否为空
        if extract_text:
            text += extract_text
            # split是因为每一页会有多个段
            page_numbers.extend([pn] * len(extract_text.split('\n'))) # 添加每段数据的页码
    return text,page_numbers
        
pdf = fitz.open('aaa.pdf')
text, page_numbers = extract_text_pdf(pdf)
print(text)
print(len(page_numbers))
pdf.close()

目
录
（可点击跳转）
序----------------------------------------------------------------- 1
第一章
错失良机--------------------------------------------------- 6
第二章
曲线买船-------------------------------------------------- 17
第三章
暗度陈仓-------------------------------------------------- 28
第四章
步步惊心-------------------------------------------------- 39
第五章
幕后推手-------------------------------------------------- 49
第六章
出击避险-------------------------------------------------- 66
第七章
股权纷争-------------------------------------------------- 83
第八章
心急如焚-------------------------------------------------- 96
第九章
黑海船厂------------------------------------------------- 110
第十章
胆大妄为------------------------------------------------- 128
第十一章
阴差阳错----------------------------------------------- 154
第十二章
移交国家----------------------------------------------- 176
第十三章
不幸功臣----------------------------------------------- 187
第十四章
合力共振----------------------------------------------- 199
关于《“瓦良格”号航母来中国》一书连载情况的说明----------------- 217
附录一：本书正文涉

In [2]:
# 创建文档对象
from langchain.docstore.document import Document

documents = [
    Document(page_content=text, metadata={'source':'aaa.pdf'})
]

In [3]:
# 数据分块
from langchain.text_splitter import RecursiveCharacterTextSplitter # 对文本进行拆分

# 中文分块
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50, # 适合中文段落（段落大小）
    chunk_overlap=0, # 避免上下文缺失(保持段落完整性)
    separators=["\n\n", "\n", "。", "！", "？"],
)
splits = text_splitter.split_documents(documents)

In [4]:
# 选择文本嵌入模型
from langchain_huggingface import HuggingFaceEmbeddings

# 加载本地模型
embeddings = HuggingFaceEmbeddings(
    model_name='tmp/bge-small-zh-v1.5',
    model_kwargs={'device':'cpu'},
    encode_kwargs={'normalize_embeddings': False} # 提升相似度计算精度
)

In [5]:
# 初始化向量数据库
from langchain_community.vectorstores import Chroma

db = Chroma.from_documents(
    documents=splits,               # 输入的文本
    embedding=embeddings,           # 选择模型
    persist_directory="./tmp/chroma_db3" # 数据库存储的目录
)


In [6]:
# 加载大语言模型

import os
from langchain_community.chat_models import ChatZhipuAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
import warnings

warnings.filterwarnings("ignore", message=".*HMAC key is 16 bytes long.*")

key = os.getenv('ZHIPU_API_KEY')
os.environ["ZHIPUAI_API_KEY"] = key

chat = ChatZhipuAI(
    model="glm-4",
    temperature=0.5,
)

In [10]:
# 结合LLM模型使用
from langchain.chains.retrieval_qa.base import RetrievalQA
# from langchain.chains import RetrievalQA 也可以这么简写

# 创建一个RAG链
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=db.as_retriever(search_kwargs={'k': 1}), # 检索几段内容
    return_source_documents=True # 返回源文档（调式用的）
)

# 测试
query = "《“瓦良格”号航母来中国》中提到的国企华夏证券公司董事长兼总裁是谁？"
result = qa_chain({'query': query})
print('答案：', result['result'])
print('来源：', result['source_documents'][0].page_content)

答案： 根据您提供的上下文，华夏证券公司董事长兼总裁是邵淳。文中提到他是促使"瓦良格"号来中国的重要人物。
来源： 华夏证券公司董事长兼总裁邵淳，是作者描绘的重要人物。他是促使“瓦良格”来中国的真


In [12]:
# 如果问一下不包括在知识库中的问题，大模型就会不知道怎么回答，所以需要下面的代码来创建一个提示词
from langchain.prompts import PromptTemplate

# 定义提示模板
prompt_template = '''
你是一个问答机器人。
你得任务是根据下述给定的已知信息回答用户问题。

已知信息：
{context} # 检索出来的原始文档

用户问题：
{question} # 用户的问题

如果已知信息中不包含用户问题的答案，或者已知信息无法回答用户问题，请直接返回“俺不知道！你找别的人吧！”。
请不要输出已知信息中不包含的信息或答案。
请用中文回答用户问题。
'''

prompt = PromptTemplate(
    template = prompt_template,
    input_variables = ['context', 'question']
)

In [13]:
# 创建一个RAG链
qa_chain = RetrievalQA.from_chain_type(
    llm=chat,
    retriever=db.as_retriever(search_kwargs={'k': 1}), # 检索几段内容
    return_source_documents=True, # 返回源文档（调式用的）
    chain_type_kwargs={'prompt': prompt} # 自定义提示词
)

In [14]:
query = "DeepSeek是什么？"
result = qa_chain({'query': query})
print('答案：', result['result'])

答案： 俺不知道！你找别的人吧！
